In [ ]:
"""
老版本的训练脚本，供基本循环参考。
"""
import os
import sys
from types import SimpleNamespace
from collections import defaultdict
from datetime import datetime
from IPython.display import display, clear_output
from torch.optim import Adam
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from mpl_toolkits.mplot3d import Axes3D
from tqdm.notebook import tqdm

# 导入项目模块
from drone_env import DroneSimulator
from model import Model
from loss import DroneLoss
from random import normalvariate
# 从 train.py 导入训练器
from train import DroneTrainer, is_save_iter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# 一个基本的训练循环示例，现实告诉我们这里有梯度爆炸问题。
B = 4  # 批量大小
dt = 0.05  # 时间步长基准
num_steps = 200  # 模拟步数

# 初始化仿真环境
# mesh_path: 从 ipynb 目录看，数据在 ../data
# 完全暴露参数的版本
env = DroneSimulator(
    batch_size=B, 
    dt=dt, 
    mesh_path="./data/sample/sample4.obj", 
    image_size=(48,64),
    
    device=device ,
    enable_airmode=True,
    enable_induced_drag=False,
    noise_std=0.04,
    grad_decay=0.5,
    num_samples= 100000
)
model = Model(dim_obs=7, dim_action=6).to(device) # 初始化模型

optim = Adam(model.parameters(), lr=1e-4) # 使用 Adam 优化器

# 随机生成目标位置
target_pos = torch.rand(B, 3, device=device) * 2.0  # 目标位置，还缺少每个批次中的再随机化
state = env.reset()
hx =None # GRU 隐藏状态初始化为 None

# 个体随机化 max_speed (参考项目逻辑：0.75 + 2.5 * rand)
# shape: (B, 1) 以便后续直接广播
max_speed = (0.75 + 2.5 * torch.rand((B, 1), device=device))

# 初始化损失计算器
losser = DroneLoss(ctl_dt= dt)
p_history = []
v_history = []
target_vel_history = [] # 目标速度记录
act_history = [] # 动作历史
vec_to_obj_history = []
v_preds = [] # 记录速度预测值。

for step in range(num_steps):
    # 随机化控制间隔
    current_dt = normalvariate(dt, dt * 0.1)
    # 添加当前状态
    p_history.append(env.p)
    v_history.append(env.v)
    vec_to_obj_history.append(env.vec_to_obj())
    with torch.no_grad():
        rgb_images, depth_images = env.render(camera_pitch=10.0,return_tensor=True,return_rgb=False, dt=current_dt)
    depth_images = depth_images.requires_grad_(False)  # 确保深度图不需要梯度
    
    # rgb_images = rgb_images.requires_grad_(False)  # 确保 RGB 图不需要梯度，这里暂时不需要 RGB 图，所以注释掉。
    # 预处理深度图像
    depth_images = depth_images.clamp(0.3,24.0)  # 限制深度范围，去除无效值
    x = 3.0/depth_images -0.6  # 转换为近似线性空间
    # 是否添加噪声（暂时默认添加，定稿时请添加相关逻辑。）
    x = x + torch.randn_like(x) * 0.02 # 暂时默认噪声系数0.02,定稿时请添加相关逻辑。
    x = F.max_pool2d(x.unsqueeze(1),kernel_size=4,stride=4)

    # 计算目标向量
    # 使用 detach() 断开当前位置的梯度，防止模型试图通过瞬移来缩小目标距离
    target_vec_raw = target_pos - env.p.detach()  
    target_vec_norm = torch.norm(target_vec_raw,2,-1,keepdim=True) 
    target_vec_unit = target_vec_raw / (target_vec_norm + 1e-6)
    
    # 限制最大速度 (max_speed 现在是 tensor (B,1))
    target_vec_limited = target_vec_unit * torch.minimum(target_vec_norm, max_speed)
    
    # 记录目标速度向量
    target_vel_history.append(target_vec_limited)
    # 转换到机体坐标系
    target_vec_body = torch.squeeze(target_vec_limited.unsqueeze(1) @ env.R, 1)
    body_z_axis = env.R[:,2] # 提取机体 Z 轴
    obs_state = torch.cat([target_vec_body, body_z_axis, env.margin.unsqueeze(1)], dim=-1) # (B, 7)
    # 模型推理
    pre,_,hx = model(x=x,v=obs_state,hx=hx) # 前向传播，获得动作指令 （注：我暂时不清楚为什么返回会有个none，但先这样写着，参考项目组应该有他的大病)
    # 输出pre是（B，6），前三维是机体坐标系下的期望加速度，后三维是模型对自己速度的估计

    # 将输出 reshape 为 (B, 3, 2)，其中 dim=2 分别对应 [加速度向量, 速度向量] (在机体坐标系下)
    # 修正逻辑：先按 (B, 2, 3) 分离 加速度(3) 和 速度(3)，再 permute 到 (B, 3, 2)
    # 这样 dim=2 的两个通道才是对应的 [acc, vel]
    # 左乘 env.R (B, 3, 3) 将这两个向量从机体坐标系旋转回世界坐标系
    # 最后 unbind 分离出 a_pred (预测加速度) 和 v_pred (预测速度) - 均为世界坐标系
    a_pred, v_pred = (env.R @ pre.reshape(B, 2, 3).permute(0, 2, 1)).unbind(-1)
    # 记录动作和预测速度
    v_preds.append(v_pred)  # 记录速度预测值
    act = a_pred 
    act_history.append(act)
    # 步进
    state = env.step(act_cmd=act, target_pos_vector=target_vec_raw, dt=current_dt)

# 循环结束，计算损失
p_history = torch.stack(p_history) # (num_steps, B, 3)
v_history = torch.stack(v_history) # (num_steps, B, 3)
target_vel_history = torch.stack(target_vel_history) # (num_steps, B, 3)
act_history = torch.stack(act_history) # (num_steps, B, 3)
vec_to_obj_history = torch.stack(vec_to_obj_history) # (num_steps, B, 3)
v_preds = torch.stack(v_preds) # (num_steps, B, 3)

# 计算损失
loss , metrics = losser.forward(
    p_history = p_history,
    v_history = v_history,
    target_vel_history = target_vel_history,
    act_history = act_history,
    vec_to_obj_history = vec_to_obj_history,
    v_preds = v_preds,
    env_margin = env.margin,
    env_g_std = None
)

# 反向传播和优化步骤
optim.zero_grad() # 清除梯度
loss.backward() # 反向传播
optim.step() # 优化器更新参数   


# 输出损失和指标
print(f"Loss: {loss.item():.4f}")
print("Metrics:", metrics)

Loading mesh from: ./data/sample/sample4.obj
Loss: 5.6815
Metrics: {'loss_ground_affinity': tensor(1.7113, device='cuda:0', grad_fn=<MeanBackward0>), 'loss_v': tensor(1.7851, device='cuda:0', grad_fn=<SmoothL1LossBackward0>), 'loss_v_pred': tensor(0.0743, device='cuda:0', grad_fn=<MseLossBackward0>), 'loss_bias': tensor(0.0315, device='cuda:0', grad_fn=<MulBackward0>), 'loss_d_acc': tensor(5.0628e-06, device='cuda:0', grad_fn=<MeanBackward0>), 'loss_d_jerk': tensor(2.8819e-06, device='cuda:0', grad_fn=<MeanBackward0>), 'loss_d_snap': tensor(9.1241e-06, device='cuda:0', grad_fn=<MeanBackward0>), 'loss_obj_avoidance': tensor(0.8425, device='cuda:0', grad_fn=<MeanBackward0>), 'loss_collide': tensor(1.2420, device='cuda:0', grad_fn=<MeanBackward0>), 'loss_speed': tensor(1.8555, device='cuda:0', grad_fn=<SmoothL1LossBackward0>), 'success_rate': 0.0, 'avg_speed': 0.4274461269378662, 'max_speed': 0.7848283648490906}
